# Aula 11 - AutoML com PyCaret

**Módulo 03 IN** - Lógica para predição com inteligência artificial
**24/09/2026 - Sprint 4 - Prof. Ovidio Lopes da Cruz Netto**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/canaldoovidio/2026-2A-M03/blob/main/notebooks/aula11.ipynb)

## O que este notebook é

A Aula 10 gastou trinta minutos ajustando os hiperparâmetros de uma floresta e a levou de 5,04%
para 4,21% de MAPE. Este notebook roda um comparador automático de candidatos sobre a mesma base,
em quatro segundos, e a pergunta da aula é se ele bate esse resultado.

A resposta curta é que bate, e por um motivo que vale mais que o número: **o AutoML varre a família
de modelos, e o ajuste de hiperparâmetros não**. Varrer a família expôs uma decisão que o acervo
tomou na Aula 07, por um motivo correto, e nunca reexaminou quando a família mudou.

A resposta longa tem quatro medições:

1. **o quadro de referência**, com os quatro modelos que o acervo já tem, todos nos mesmos 24 meses
   de teste;
2. **o leaderboard**, com 22 candidatos ordenados em segundos;
3. **como ler esse leaderboard sem se enganar**, começando pelo R2, que despenca de 0,98 para 0,48
   sem o modelo piorar;
4. **o que o PyCaret não decide por você**: o alvo, o protocolo de validação e as features.

Os números batem com `tools/tests/test_automl_aula11.py`, e o motivo da versão escolhida do PyCaret
está na `ADR-014`.

## 1. A base e os quatro modelos de referência

A célula abaixo remonta a base analítica das Aulas 07 a 10 e instala o PyCaret na versão exata que
a aula usa.

**Sobre a versão.** Os autoestudos desta semana ensinam `setup()` e `compare_models()`, que é a API
do PyCaret 3. Esta aula usa o PyCaret 4.0.0a8, porque o 3.3.2 **recusa o Python 3.12 na
importação**, com uma checagem explícita dentro do pacote, e porque instalá-lo rebaixaria
`scikit-learn`, `numpy` e `pandas` para versões anteriores às dos outros dez notebooks do acervo. A
tabela de conversão entre as duas APIs está na seção 2, e o motivo completo na `ADR-014`.

In [ ]:
import calendar
import os
import subprocess
import sys
import urllib.request
import warnings

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

VERSAO_PYCARET = "4.0.0a8"
try:
    import pycaret
    assert pycaret.__version__ == VERSAO_PYCARET
except (ImportError, AssertionError):
    print("instalando o pycaret %s (demora alguns minutos na primeira vez)" % VERSAO_PYCARET)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "pycaret==" + VERSAO_PYCARET])
    import pycaret

print("pycaret", pycaret.__version__)

SERIES = [
    "abate_bovinos",
    "abate_suinos",
    "abate_frangos",
    "producao_ovos",
    "producao_leite",
]
ALVO = "abate_frangos"
FEATURES = (["lag1", "lag2", "lag3", "lag12", "sen", "cos", "dias"]
            + [serie + "_lag1" for serie in SERIES if serie != ALVO])
N_TESTE = 24
SEMENTE = 42

MENSAL_LOCAL = os.path.join("..", "dados", "mensal")
MENSAL_BRUTA = ("https://raw.githubusercontent.com/canaldoovidio/2026-2A-M03/"
                "main/dados/mensal/")

caminhos = {}
for nome in SERIES:
    arquivo = nome + ".csv"
    local = os.path.join(MENSAL_LOCAL, arquivo)
    if os.path.exists(local):
        caminhos[nome] = local
    else:
        if not os.path.exists(arquivo):
            try:
                urllib.request.urlretrieve(MENSAL_BRUTA + arquivo, arquivo)
            except Exception as erro:
                raise RuntimeError(
                    "Nao foi possivel baixar '%s' pela internet (%s). "
                    "Se a rede da sala falhou, peca a pasta 'dados' para uma dupla "
                    "que tenha o repositorio clonado no computador (ela fica na raiz "
                    "do repositorio) e coloque essa pasta ao lado deste notebook. "
                    "Depois, rode esta celula de novo." % (arquivo, erro)
                ) from erro
        caminhos[nome] = arquivo

base = None
for nome in SERIES:
    coluna = (pd.read_csv(caminhos[nome])[["periodo", "valor"]]
              .rename(columns={"valor": nome}))
    base = coluna if base is None else base.merge(coluna, on="periodo", how="inner")
base = base.sort_values("periodo").reset_index(drop=True)

base["mes"] = base["periodo"].str[-2:].astype(int)
base["dias"] = [calendar.monthrange(int(p[:4]), int(p[-2:]))[1] for p in base["periodo"]]
base["sen"] = np.sin(2 * np.pi * base["mes"] / 12)
base["cos"] = np.cos(2 * np.pi * base["mes"] / 12)
for k in (1, 2, 3, 12):
    base["lag%d" % k] = base[ALVO].shift(k)
for nome in SERIES:
    if nome != ALVO:
        base[nome + "_lag1"] = base[nome].shift(1)
base = base.dropna().reset_index(drop=True)

CORTE = len(base) - N_TESTE
X = base[FEATURES]
y = base[ALVO].to_numpy(dtype=float)
lag12 = base["lag12"].to_numpy(dtype=float)
razao = y / lag12

Xtr, Xte = X.iloc[:CORTE], X.iloc[CORTE:]
yte, lag12te = y[CORTE:], lag12[CORTE:]

print("linhas: %d   treino: %d meses   teste: %d meses (%s a %s)"
      % (len(base), CORTE, N_TESTE, base["periodo"].iloc[CORTE], base["periodo"].iloc[-1]))

Antes de rodar qualquer coisa automática, é preciso saber contra o que comparar. O acervo já tem
quatro modelos medidos sobre exatamente estes 24 meses, e a célula abaixo os reúne num quadro só.

É a primeira vez que eles aparecem lado a lado, e a ordem surpreende.

In [ ]:
def mape(real, previsto):
    return float(np.mean(np.abs((real - previsto) / real)) * 100)


escalador = StandardScaler().fit(Xtr)
Ztr, Zte = escalador.transform(Xtr), escalador.transform(Xte)

linear_nivel = LinearRegression().fit(Ztr, y[:CORTE])
linear_razao = LinearRegression().fit(Ztr, razao[:CORTE])
floresta = RandomForestRegressor(n_estimators=600, max_depth=4, min_samples_leaf=5,
                                 random_state=SEMENTE).fit(Xtr, razao[:CORTE])
fator = float(np.mean(razao[:CORTE]))

referencias = {
    "regressao linear, alvo em nivel": mape(yte, linear_nivel.predict(Zte)),
    "regressao linear, alvo em razao": mape(yte, linear_razao.predict(Zte) * lag12te),
    "baseline de coeficiente fixo da LDC": mape(yte, lag12te * fator),
    "floresta ajustada na Aula 10": mape(yte, floresta.predict(Xte) * lag12te),
}

print("%-38s %s" % ("modelo", "MAPE nos 24 meses de teste"))
for nome, erro in sorted(referencias.items(), key=lambda item: item[1]):
    print("%-38s %20.2f%%" % (nome, erro))

O modelo que **duas aulas de trabalho manual produziram é o pior dos quatro**. A floresta ajustada
na Aula 10 erra 4,21%, e perde para a baseline de coeficiente fixo que a própria LDC já usava
(3,71%), que por sua vez perde para a regressão linear do fecho da Aula 07 (3,32%).

Isso não significa que as Aulas 07 a 10 foram perdidas. Significa que elas trabalharam dentro de
uma escolha de família que ninguém reabriu: a Aula 07 escolheu árvore e ensemble, a Aula 10 ajustou
os hiperparâmetros da floresta, e **hiperparâmetro não muda a família do modelo**.

Guarde também a primeira linha, a regressão linear sobre o alvo **em nível**. Ela nunca apareceu em
nenhuma aula, e é a melhor das quatro.

## 2. O leaderboard, em quatro segundos

O `compare_models()` treina um catálogo inteiro de famílias sob a mesma validação e devolve a lista
ordenada. A conversão entre a API dos autoestudos e a desta aula é esta:

| autoestudo (PyCaret 3) | aula (PyCaret 4.0.0a8) |
| --- | --- |
| `setup(data=df, target="alvo")` | `exp = RegressionExperiment(); exp.fit(X, y)` |
| `compare_models()` | `exp.compare_models()` |
| `pull()` | `exp.pull()` |

Os argumentos de configuração, que no PyCaret 3 ficavam no `setup()`, passam para o construtor do
`RegressionExperiment`. É onde vivem os dois que esta aula vai examinar: `train_size` e
`fold_strategy`.

In [ ]:
from pycaret.regression import RegressionExperiment
import time


def rodar(alvo, **kwargs):
    """Roda o compare_models sobre os 315 meses de treino e devolve o essencial."""
    inicio = time.time()
    exp = RegressionExperiment(session_id=SEMENTE, **kwargs)
    exp.fit(Xtr, pd.Series(alvo[:CORTE], name="alvo"))
    resultado = exp.compare_models(n_select=5)
    decorrido = time.time() - inicio
    quadro = exp.pull()
    return {"exp": exp, "quadro": quadro, "ids": list(resultado.ranked_ids),
            "modelos": resultado.models, "segundos": decorrido,
            "treino_interno": len(exp.X_train), "teste_interno": len(exp.X_test)}


em_razao = rodar(razao)

print("candidatos avaliados: %d" % len(em_razao["quadro"]))
print("tempo: %.0f segundos" % em_razao["segundos"])
print()
print(em_razao["quadro"].head(10).to_string())

Vinte e dois candidatos, poucos segundos, e o topo inteiro é de **modelos lineares**: `ridge`,
`lar`, `tr`, `br`, `lasso`, `lr`, `llar` e `en`. A primeira família de outro tipo aparece só na nona
posição.

Duas coisas para reparar na tabela antes de seguir.

A primeira é a ordem. `br` aparece acima de `lasso` com MAPE **maior** (4,04% contra 4,03%), porque
o leaderboard vem ordenado pelo **R2**, e não pela métrica que o case usa. Quem lê a primeira linha
sem olhar a coluna está lendo o ranking de outra métrica.

A segunda é o tamanho do treino, que a célula abaixo mostra.

In [ ]:
print("meses que a dupla entregou ao PyCaret: %d" % CORTE)
print("meses que ele usou para treinar:       %d" % em_razao["treino_interno"])
print("meses que ele separou por conta:       %d" % em_razao["teste_interno"])
print()
print("train_size default: %.2f" % RegressionExperiment().get_params()["train_size"])
print("fold_strategy default: %r" % RegressionExperiment().get_params()["fold_strategy"])

O construtor traz `train_size=0.7` e `fold_strategy="kfold"`, e os dois são exatamente os erros de
protocolo que a Aula 10 ensinou a reconhecer:

- **`train_size=0.7` separa 95 dos 315 meses por sorteio.** É o `train_test_split` aleatório numa
  série temporal, que a Aula 09 mediu inflando o MAPE de quem memoriza vizinho.
- **`fold_strategy="kfold"` é o `KFold`**, que a Aula 10 mostrou pondo 252 dos 252 meses de treino
  no futuro, na primeira dobra.

A ferramenta que automatiza a comparação **não** automatiza o protocolo. Ela vem com o protocolo
errado para série temporal ligado por padrão, e quem tem que desligar é a dupla.

## 3. O R2 que despenca sem o modelo piorar

Antes de consertar o protocolo, um alerta de leitura. A célula abaixo roda o mesmo
`compare_models()`, com as mesmas features e os mesmos defaults, mudando **só o alvo**: em vez da
razão sobre o mesmo mês do ano anterior, o nível em quilogramas.

In [ ]:
em_nivel = rodar(y)

print("%-22s %10s %10s" % ("alvo", "R2", "MAPE"))
for rotulo, r in (("em nivel", em_nivel), ("em razao", em_razao)):
    print("%-22s %9.4f %9.2f%%"
          % (rotulo, float(r["quadro"].iloc[0]["R2"]), float(r["quadro"].iloc[0]["MAPE"]) * 100))

O R2 cai de cerca de **0,98 para cerca de 0,48**, e o MAPE dos dois fica na mesma casa.

O modelo não piorou. O R2 é a fração da variância do alvo que o modelo explica, e os dois alvos têm
variâncias completamente diferentes: o nível cresce de 0,3 para 1,2 bilhão de quilogramas ao longo
de 28 anos, enquanto a razão fica quase toda entre 0,9 e 1,2. Explicar 98% de uma série que sobe é
fácil; explicar 48% de uma razão que oscila em torno de 1 é outra coisa.

É o mesmo erro de leitura que a Aula 09 tratou com o RMSE entre dois conjuntos de teste de escalas
diferentes: uma métrica que depende da escala do alvo não é comparável entre alvos. O MAPE é razão
entre erro e valor real, e por isso atravessa a troca sem se mover.

**Consequência prática para a ART.7:** o leaderboard vem ordenado por R2. Se a dupla comparar dois
experimentos com alvos diferentes usando a primeira linha de cada um, vai concluir que o alvo em
nível é duas vezes melhor. Ele não é.

## 4. Trocar o protocolo, e ver a estimativa melhorar

Agora o conserto. A célula abaixo roda o mesmo `compare_models()` sobre o alvo em razão, com
`fold_strategy=TimeSeriesSplit(5)` e com o `train_size` quase inteiro, para que o PyCaret use os 315
meses que a dupla entregou em vez de sortear 220 deles.

Depois, mede o campeão de cada configuração nos 24 meses de teste, que nenhuma das duas viu.

In [ ]:
temporal = rodar(razao, fold_strategy=TimeSeriesSplit(n_splits=5), train_size=0.99)
temporal_nivel = rodar(y, fold_strategy=TimeSeriesSplit(n_splits=5), train_size=0.99)


def erro_de_teste(resultado, em_razao=True):
    campeao = resultado["modelos"][0]
    previsto = np.asarray(campeao.predict(Xte), dtype=float)
    if em_razao:
        previsto = previsto * lag12te
    return mape(yte, previsto)


print("%-8s %-10s %8s %10s %9s %10s"
      % ("alvo", "protocolo", "treino", "estimado", "teste", "diferenca"))
for alvo, protocolo, r, razao_ in (("nivel", "default", em_nivel, False),
                                   ("nivel", "temporal", temporal_nivel, False),
                                   ("razao", "default", em_razao, True),
                                   ("razao", "temporal", temporal, True)):
    estimado = float(r["quadro"].iloc[0]["MAPE"]) * 100
    teste = erro_de_teste(r, em_razao=razao_)
    print("%-8s %-10s %8d %9.2f%% %8.2f%% %9.2f"
          % (alvo, protocolo, r["treino_interno"], estimado, teste, abs(estimado - teste)))

Os quatro quadrantes de alvo por protocolo. Em cada alvo, as duas configurações escolhem o mesmo
campeão e chegam a um MAPE de teste parecido. O que muda é a **qualidade da estimativa**: a
diferença entre o estimado e o medido cai de 0,82 para 0,51 no alvo em nível, e de 0,47 para 0,17 no
alvo em razão.

Repare também na coluna de treino. O protocolo default entrega ao campeão **220 dos 315 meses**,
sorteados, e o temporal entrega 311. Comparar um modelo do default com um modelo treinado nos 315
meses inteiros, como são os quatro da seção 1, compara família **e** volume de dado ao mesmo tempo.
Por isso o quadro de referência usa o campeão do protocolo temporal.

É a confirmação, dentro da ferramenta nova, do argumento que a Aula 10 defendeu como regra de
protocolo. Naquela aula, a troca de validador não melhorou o modelo, e o argumento foi que ela torna
a estimativa auditável. Aqui dá para medir o quanto: a estimativa auditável é também a estimativa
mais próxima da verdade.

## 5. O que o AutoML achou que duas aulas não acharam

Falta a pergunta de abertura: o campeão automático bate o modelo ajustado à mão?

In [ ]:
print("%-40s %s" % ("modelo", "MAPE nos 24 meses de teste"))
linhas = sorted(list(referencias.items())
                + [("campeao do AutoML, razao e temporal", erro_de_teste(temporal)),
                   ("campeao do AutoML, nivel e temporal",
                    erro_de_teste(temporal_nivel, em_razao=False))],
                key=lambda item: item[1])
for nome, erro in linhas:
    marca = "  <--" if "AutoML" in nome else ""
    print("%-40s %20.2f%%%s" % (nome, erro, marca))

print()
custo = referencias["regressao linear, alvo em razao"] - referencias["regressao linear, alvo em nivel"]
print("o alvo em razao custa %.2f ponto percentual a regressao linear" % custo)

Bate, e com folga. Mas o ganho não veio de o AutoML achar um modelo exótico: veio de ele **testar a
família linear sobre o alvo em nível**, que é uma combinação que nenhuma aula tinha rodado.

A decisão de treinar sobre a razão foi tomada na Aula 07, e foi correta pelo motivo dela: a árvore
de decisão não consegue extrapolar uma série que cresce, porque a folha devolve uma média de valores
vistos, e o alvo em razão resolve isso. Essa decisão ficou valendo para todos os modelos seguintes,
inclusive para os lineares, que **não têm esse problema**: uma regressão linear extrapola sem
dificuldade.

Para o modelo linear, o alvo em razão custa cerca de meio ponto percentual. Nenhuma busca de
hiperparâmetro encontraria isso, porque hiperparâmetro não muda o alvo. Varrer a família encontrou.

**E o alvo em razão continua certo para a árvore.** A célula abaixo confirma: a mesma floresta da
Aula 10, treinada em nível, fica pior que a treinada em razão. A decisão da Aula 07 não estava
errada; ela estava sendo aplicada fora do caso em que foi tomada.

In [ ]:
floresta_nivel = RandomForestRegressor(n_estimators=600, max_depth=4, min_samples_leaf=5,
                                       random_state=SEMENTE).fit(Xtr, y[:CORTE])

print("floresta da Aula 10, alvo em nivel: %.2f%%" % mape(yte, floresta_nivel.predict(Xte)))
print("floresta da Aula 10, alvo em razao: %.2f%%" % referencias["floresta ajustada na Aula 10"])
print()
print("regressao linear,   alvo em nivel: %.2f%%" % referencias["regressao linear, alvo em nivel"])
print("regressao linear,   alvo em razao: %.2f%%" % referencias["regressao linear, alvo em razao"])

## O que levar para a ART.7

**ART.7 Comparação de modelos, peso 8**, fecha a Sprint 4, com review em 25/09. O que esta aula
acrescenta ao protocolo das Aulas 09 e 10:

1. **varrer a família antes de ajustar o candidato**. Quatro segundos de `compare_models()` deram
   mais que trinta minutos de `GridSearchCV`, porque o ganho estava na família e no alvo, e não nos
   hiperparâmetros;
2. **declarar o alvo junto de cada número**, e reabrir a escolha de alvo quando a família mudar. A
   decisão da Aula 07 continua certa para a árvore e custa meio ponto ao modelo linear;
3. **não confiar na ordem do leaderboard**, que vem por R2. O R2 não é comparável entre alvos de
   variâncias diferentes, e a métrica que o case usa é o MAPE;
4. **desligar os dois defaults temporais do PyCaret** (`train_size` por sorteio e
   `fold_strategy="kfold"`), porque a ferramenta automatiza a comparação e não o protocolo;
5. **declarar a versão da biblioteca**. Esta aula roda no PyCaret 4.0.0a8, com uma API diferente da
   dos autoestudos, e um resultado sem versão declarada não é reproduzível.

O melhor candidato do acervo, ao fim da Sprint 4, é um modelo linear sobre o alvo em nível, e não a
floresta que duas aulas ajustaram. Na Aula 12 ele é o que vai para o `Pipeline` do scikit-learn e
para o MLflow.